# 💳 Week 6 - Part 1: Encoding Categorical Variables
### Data Science & Visualization · Team 1 · 24ADI204
**Objective:** Transform non-numeric text columns (`category`, `gender`, `state`, `job`) into numerical formats so machine learning algorithms can process them, comparing **One-Hot Encoding**, **Label Encoding**, and **Frequency Encoding**.
---
**Interconnection Pipeline Step:** `1 of 4`  
- **Input:** `fraudTrain_cleaned.csv` (from Week 3 data cleaning)  
- **Output:** `encoded_transactions.csv` (saved for Notebook 2: Normalization & Scaling)


In [ ]:
# 1. Setup & Environment Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting aesthetics
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('Libraries imported successfully!')


In [ ]:
# 2. Multi-Environment Data Loading (Local, Colab, Kaggle)
possible_paths = [
    'fraudTrain_cleaned.csv',
    'github_repo/Week 3/fraudTrain_cleaned.csv',
    '../Week 3/fraudTrain_cleaned.csv',
    'fraudTrain.csv',
    'archive (2)/fraudTrain.csv'
]

data_path = None
for p in possible_paths:
    if os.path.exists(p):
        data_path = p
        break

print(f'Loading dataset from: {data_path}')
# Load working sample (100,000 rows) for smooth interactive execution
df = pd.read_csv(data_path, nrows=100000)
print(f'Loaded {len(df):,} rows and {df.shape[1]} columns.')
df.head(3)


## Step 1: Identifying & Inspecting Categorical Variables
Categorical variables contain discrete text values. In our dataset:
- `gender`: Binary (2 unique values: M, F)
- `category`: Nominal low cardinality (14 unique merchant categories)
- `state`: Nominal medium cardinality (50 US states)
- `job`: Nominal high cardinality (400+ job titles)


In [ ]:
cat_cols = ['gender', 'category', 'state', 'job']
for col in cat_cols:
    if col in df.columns:
        print(f'{col:12} -> {df[col].nunique()} unique categories. Top 3: {df[col].value_counts().head(3).to_dict()}')


## Step 2: Binary & Label / Ordinal Encoding
1. **Binary Encoding for `gender`**: Map `F -> 0` and `M -> 1`.
2. **Ordinal / Label Encoding for `category`**: Assign each category an integer code `(0 to 13)`. This preserves dimensionality without expanding the feature space.


In [ ]:
# Gender mapping
df['gender_encoded'] = (df['gender'] == 'M').astype(int)

# Category mapping
categories = sorted(df['category'].unique())
cat_mapping = {cat: idx for idx, cat in enumerate(categories)}
df['category_code'] = df['category'].map(cat_mapping)

print('Gender Mapping: F -> 0, M -> 1')
print(f'Category Mapping Sample: {list(cat_mapping.items())[:5]}')
df[['gender', 'gender_encoded', 'category', 'category_code']].head()


## Step 3: One-Hot Encoding (OHE) for Low Cardinality
One-Hot Encoding converts each category into a separate binary column (`0` or `1`).
It avoids introducing false numerical order between categories (e.g., Grocery is not 'greater than' Entertainment).


In [ ]:
# One-Hot Encoding on 'category'
df_ohe = pd.get_dummies(df['category'], prefix='cat', drop_first=True, dtype=int)
print(f'Created {df_ohe.shape[1]} One-Hot columns.')
df_ohe.head(3)


## Step 4: Frequency / Target Encoding for High Cardinality (`state` & `job`)
Using One-Hot Encoding on `job` (400+ titles) would cause the **Curse of Dimensionality**.
Instead, we use **Frequency Encoding**: replacing each category with its percentage frequency in the dataset.


In [ ]:
# Frequency Encoding for State and Job
state_freq = df['state'].value_counts(normalize=True).to_dict()
df['state_freq'] = df['state'].map(state_freq)

job_freq = df['job'].value_counts(normalize=True).to_dict()
df['job_freq'] = df['job'].map(job_freq)

print('Frequency encoding applied successfully!')
df[['state', 'state_freq', 'job', 'job_freq']].head(3)


## Step 5: Visualizing Encoded Category Distributions
Let us visualize the encoded categorical frequencies across merchant categories.


In [ ]:
plt.figure(figsize=(10, 4))
df['category'].value_counts().plot(kind='bar', color='#3498db')
plt.title('Category Frequencies in Encoded Dataset', fontweight='bold')
plt.xlabel('Merchant Category')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## Step 6: Saving Interconnected Dataset for Notebook 2
We assemble the cleaned numeric fields and our newly encoded categorical features into `encoded_transactions.csv`.


In [ ]:
# Ensure temporal and numerical features exist
if 'customer_age' not in df.columns and 'dob' in df.columns:
    df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
    df['dob'] = pd.to_datetime(df['dob'])
    df['customer_age'] = (df['trans_date_trans_time'] - df['dob']).dt.days // 365
    df['transaction_hour'] = df['trans_date_trans_time'].dt.hour

cols_to_save = [
    'amt', 'city_pop', 'lat', 'long', 'merch_lat', 'merch_long',
    'transaction_hour', 'customer_age', 'gender_encoded', 'category_code',
    'state_freq', 'job_freq', 'is_fraud'
]
final_encoded_df = df[cols_to_save].copy()
output_file = 'encoded_transactions.csv'
final_encoded_df.to_csv(output_file, index=False)
print(f'Successfully saved interconnected dataset to: {output_file} (Shape: {final_encoded_df.shape})')
